In [23]:
use std::collections::HashMap;

// ============================================================
// 1. トークン & 命令定義
// ============================================================
#[derive(Debug, Clone, PartialEq)]
enum Token { S, T, N }

#[derive(Debug, Clone)]
enum Instr {
    Push(i64), Dup, Swap, Discard, Copy(usize), Slide(usize),
    Add, Sub, Mul, Div, Mod,
    Store, Load,
    Label(String), Call(String), Jump(String),
    Jz(String), Jneg(String), Ret, End,
    Putc, Putn, Readc, Readi,
}

// ============================================================
// 2. ソースフィルタ: space/tab/newline のみ抽出
// ============================================================
fn filter_source(src: &str) -> Vec<Token> {
    src.chars().filter_map(|c| match c {
        ' '  => Some(Token::S),
        '\t' => Some(Token::T),
        '\n' => Some(Token::N),
        _    => None,
    }).collect()
}

// ============================================================
// 3. 数値パース (S=0bit, T=1bit, N=終端, 先頭S=正/T=負)
// ============================================================
fn parse_number(tokens: &[Token], pos: &mut usize) -> i64 {
    let sign: i64 = match tokens.get(*pos) {
        Some(Token::T) => { *pos += 1; -1 }
        _              => { *pos += 1;  1 }
    };
    let mut val: i64 = 0;
    loop {
        match tokens.get(*pos) {
            Some(Token::N) | None => { *pos += 1; break; }
            Some(Token::S)        => { val = val * 2;     *pos += 1; }
            Some(Token::T)        => { val = val * 2 + 1; *pos += 1; }
        }
    }
    val * sign
}

// parse_label: ws_core.js の parseArg と同じ動作
// ラベルは生トークン文字列 (' '=S, '\t'=T を含む \n 終端まで) をそのまま保持
fn parse_label(tokens: &[Token], pos: &mut usize) -> String {
    let mut label = String::new();
    loop {
        match tokens.get(*pos) {
            None => break,
            Some(Token::N) => { label.push('\n'); *pos += 1; break; }  // 終端\nも含める
            Some(Token::S) => { label.push(' ');  *pos += 1; }
            Some(Token::T) => { label.push('\t'); *pos += 1; }
        }
    }
    label
}

// ============================================================
// 4. パーサー: トークン列 → 命令列
// ============================================================
fn parse_ws(tokens: &[Token]) -> Vec<Instr> {
    let mut pos = 0usize;
    let mut instrs: Vec<Instr> = Vec::new();
    macro_rules! next {
        () => {{ let t = tokens.get(pos).cloned(); pos += 1; t }};
    }
    macro_rules! num { () => {{ parse_number(tokens, &mut pos) }}; }
    macro_rules! lbl { () => {{ parse_label(tokens,  &mut pos) }}; }
    while pos < tokens.len() {
        let instr = match next!() {
            // [S] Stack Manipulation
            Some(Token::S) => match next!() {
                Some(Token::S) => Instr::Push(num!()),
                Some(Token::N) => match next!() {
                    Some(Token::S) => Instr::Dup,
                    Some(Token::T) => Instr::Swap,
                    Some(Token::N) => Instr::Discard,
                    _ => break,
                },
                Some(Token::T) => match next!() {
                    Some(Token::S) => Instr::Copy(num!() as usize),
                    Some(Token::N) => Instr::Slide(num!() as usize),
                    _ => break,
                },
                _ => break,
            },
            // [T] Arithmetic / Heap / IO
            Some(Token::T) => match next!() {
                Some(Token::S) => match next!() {
                    Some(Token::S) => match next!() {
                        Some(Token::S) => Instr::Add,   // T S S S
                        Some(Token::T) => Instr::Sub,   // T S S T
                        Some(Token::N) => Instr::Mul,   // T S S N
                        _ => break,
                    },
                    Some(Token::T) => match next!() {
                        Some(Token::S) => Instr::Div,   // T S T S
                        Some(Token::T) => Instr::Mod,   // T S T T
                        _ => break,
                    },
                    _ => break,
                },
                Some(Token::T) => match next!() {
                    Some(Token::S) => Instr::Store,     // T T S
                    Some(Token::T) => Instr::Load,      // T T T
                    _ => break,
                },
                Some(Token::N) => match next!() {
                    Some(Token::S) => match next!() {
                        Some(Token::S) => Instr::Putc,  // T N S S
                        Some(Token::T) => Instr::Putn,  // T N S T
                        _ => break,
                    },
                    Some(Token::T) => match next!() {
                        Some(Token::S) => Instr::Readc, // T N T S
                        Some(Token::T) => Instr::Readi, // T N T T
                        _ => break,
                    },
                    _ => break,
                },
                _ => break,
            },
            // [N] Flow Control
            Some(Token::N) => match next!() {
                Some(Token::S) => match next!() {
                    Some(Token::S) => Instr::Label(lbl!()),
                    Some(Token::T) => Instr::Call(lbl!()),
                    Some(Token::N) => Instr::Jump(lbl!()),
                    _ => break,
                },
                Some(Token::T) => match next!() {
                    Some(Token::S) => Instr::Jz(lbl!()),
                    Some(Token::T) => Instr::Jneg(lbl!()),
                    Some(Token::N) => Instr::Ret,
                    _ => break,
                },
                Some(Token::N) => match next!() {
                    Some(Token::N) => Instr::End,  // N N N = end
                    _ => break,
                },
                None => break,
            },
            _ => break,
        };
        instrs.push(instr);
    }
    instrs
}

// ============================================================
// 5. VM: 命令列を実行して出力を返す
// ============================================================
fn run_vm(instrs: &[Instr], input: &str) -> String {
    let mut label_map: HashMap<String, usize> = HashMap::new();
    for (i, instr) in instrs.iter().enumerate() {
        if let Instr::Label(lbl) = instr {
            label_map.insert(lbl.clone(), i);
        }
    }
    let mut stack:      Vec<i64>         = Vec::new();
    let mut heap:       HashMap<i64,i64> = HashMap::new();
    let mut call_stack: Vec<usize>       = Vec::new();
    let mut output = String::new();
    let mut input_lines = input.lines();
    let mut input_chars = input.chars();
    let mut pc = 0usize;
    
    // ---- デバッグ: ラベルマップと命令列を確認 ----
    println!("=== label_map ===");
    for (k, v) in &label_map {
        println!("  key={:?}  idx={}", k, v);
    }
    println!("=== instrs ===");
    for (i, instr) in instrs.iter().enumerate() {
        println!("  [{:02}] {:?}", i, instr);
    }
    
    while pc < instrs.len() {
        let instr = instrs[pc].clone();
        pc += 1;
        match instr {
            Instr::Push(n)   => stack.push(n),
            Instr::Dup       => { let v = *stack.last().unwrap(); stack.push(v); }
            Instr::Swap      => { let l = stack.len(); stack.swap(l-1, l-2); }
            Instr::Discard   => { stack.pop(); }
            Instr::Copy(n)   => { let l = stack.len(); let v = stack[l-1-n]; stack.push(v); }
            Instr::Slide(n)  => {
                let top = stack.pop().unwrap();
                let l = stack.len();
                stack.truncate(l.saturating_sub(n));
                stack.push(top);
            }
            Instr::Add  => { let (b,a)=(stack.pop().unwrap(),stack.pop().unwrap()); stack.push(a+b); }
            Instr::Sub  => { let (b,a)=(stack.pop().unwrap(),stack.pop().unwrap()); stack.push(a-b); }
            Instr::Mul  => { let (b,a)=(stack.pop().unwrap(),stack.pop().unwrap()); stack.push(a*b); }
            Instr::Div  => { let (b,a)=(stack.pop().unwrap(),stack.pop().unwrap()); stack.push(a/b); }
            Instr::Mod  => { let (b,a)=(stack.pop().unwrap(),stack.pop().unwrap()); stack.push(a%b); }
            Instr::Store => {
                let val  = stack.pop().unwrap();
                let addr = stack.pop().unwrap();
                heap.insert(addr, val);
            }
            Instr::Load  => {
                let addr = stack.pop().unwrap();
                stack.push(*heap.get(&addr).unwrap_or(&0));
            }
            Instr::Label(_)  => {}
            Instr::Call(lbl) => { call_stack.push(pc); pc = label_map[&lbl]; }
            Instr::Jump(lbl) => { pc = label_map[&lbl]; }
            Instr::Jz(lbl)   => { let v=stack.pop().unwrap(); if v==0 { pc=label_map[&lbl]; } }
            Instr::Jneg(lbl) => { let v=stack.pop().unwrap(); if v<0  { pc=label_map[&lbl]; } }
            Instr::Ret       => { pc = call_stack.pop().unwrap(); }
            Instr::End       => break,
            Instr::Putc  => { let c=stack.pop().unwrap() as u8 as char; output.push(c); }
            Instr::Putn  => { let n=stack.pop().unwrap(); output.push_str(&n.to_string()); }
            Instr::Readc => {
                let addr = stack.pop().unwrap();
                heap.insert(addr, input_chars.next().unwrap_or('\0') as i64);
            }
            Instr::Readi => {
                let addr = stack.pop().unwrap();
                let n: i64 = input_lines.next().unwrap_or("0").trim().parse().unwrap_or(0);
                heap.insert(addr, n);
            }
        }
    }
    output
}

// ============================================================
// 6. Whitespaceプログラムをトークン文字列として構築
//    (S=space, T=tab, N=newline)
// ============================================================
fn ws_number(n: i64) -> String {
    let mut s = String::new();
    s.push(if n >= 0 { ' ' } else { '\t' }); // 符号ビット
    let mag = n.unsigned_abs();
    if mag == 0 { s.push('\n'); return s; }
    for b in format!("{:b}", mag).chars() {
        s.push(if b == '0' { ' ' } else { '\t' });
    }
    s.push('\n');
    s
}

fn build_ksnctf7() -> Vec<Instr> {
    let mut v: Vec<Instr> = Vec::new();
    // "PIN: "
    v.push(Instr::Push(80));  v.push(Instr::Putc);
    v.push(Instr::Push(73));  v.push(Instr::Putc);
    v.push(Instr::Push(78));  v.push(Instr::Putc);
    v.push(Instr::Push(58));  v.push(Instr::Putc);
    v.push(Instr::Push(32));  v.push(Instr::Putc);
    // PIN入力・比較
    v.push(Instr::Push(0));   v.push(Instr::Readi);
    v.push(Instr::Push(0));   v.push(Instr::Load);
    v.push(Instr::Push(33355524)); v.push(Instr::Sub);
    v.push(Instr::Jz("label_0".to_string()));
    // 不正解
    v.push(Instr::Push(78));  v.push(Instr::Putc);
    v.push(Instr::Push(79));  v.push(Instr::Putc);
    v.push(Instr::End);
    // label_0: 正解 → "OK\n"
    v.push(Instr::Label("label_0".to_string()));
    v.push(Instr::Push(79));  v.push(Instr::Putc);
    v.push(Instr::Push(75));  v.push(Instr::Putc);
    v.push(Instr::Push(10));  v.push(Instr::Putc);
    // FLAG文字をスタック演算で生成
    v.push(Instr::Push(0));   v.push(Instr::Load);
    v.push(Instr::Push(33355454)); v.push(Instr::Sub);
    v.push(Instr::Dup); v.push(Instr::Putc);   // F(70)
    v.push(Instr::Push(6));  v.push(Instr::Add);
    v.push(Instr::Dup); v.push(Instr::Putc);   // L(76)
    v.push(Instr::Push(11)); v.push(Instr::Sub);
    v.push(Instr::Dup); v.push(Instr::Putc);   // A(65)
    v.push(Instr::Push(6));  v.push(Instr::Add);
    v.push(Instr::Dup); v.push(Instr::Putc);   // G(71)
    v.push(Instr::Push(24)); v.push(Instr::Add);
    v.push(Instr::Dup); v.push(Instr::Putc);   // _(95)
    v.push(Instr::Push(26)); v.push(Instr::Sub);
    v.push(Instr::Dup); v.push(Instr::Putc);   // E(69)? 確認用
    v.push(Instr::Push(40)); v.push(Instr::Add);
    v.push(Instr::Dup); v.push(Instr::Putc);
    v.push(Instr::Push(25)); v.push(Instr::Sub);
    v.push(Instr::Dup); v.push(Instr::Putc);
    v.push(Instr::Push(36)); v.push(Instr::Add);
    v.push(Instr::Dup); v.push(Instr::Putc);
    v.push(Instr::Push(66)); v.push(Instr::Sub);
    v.push(Instr::Dup); v.push(Instr::Putc);
    v.push(Instr::Push(16)); v.push(Instr::Add);
    v.push(Instr::Dup); v.push(Instr::Putc);
    v.push(Instr::Push(14)); v.push(Instr::Add);
    v.push(Instr::Dup); v.push(Instr::Putc);
    v.push(Instr::Push(14)); v.push(Instr::Add);
    v.push(Instr::Dup); v.push(Instr::Putc);
    v.push(Instr::Push(27)); v.push(Instr::Sub);
    v.push(Instr::Dup); v.push(Instr::Putc);
    v.push(Instr::Push(5));  v.push(Instr::Add);
    v.push(Instr::Dup); v.push(Instr::Putc);
    v.push(Instr::Push(29)); v.push(Instr::Add);
    v.push(Instr::Dup); v.push(Instr::Putc);
    v.push(Instr::Push(4));  v.push(Instr::Sub);
    v.push(Instr::Dup); v.push(Instr::Putc);
    v.push(Instr::Push(4));  v.push(Instr::Add);
    v.push(Instr::Dup); v.push(Instr::Putc);
    v.push(Instr::Push(28)); v.push(Instr::Sub);
    v.push(Instr::Dup); v.push(Instr::Putc);
    v.push(Instr::Push(22)); v.push(Instr::Add);
    v.push(Instr::Dup); v.push(Instr::Putc);
    v.push(Instr::Push(34)); v.push(Instr::Sub);
    v.push(Instr::Dup); v.push(Instr::Putc);
    v.push(Instr::Push(55)); v.push(Instr::Sub);
    v.push(Instr::Dup); v.push(Instr::Putc);
    v.push(Instr::End);
    v
}
// ============================================================
// 7. 実行
// ============================================================
//let instrs = build_ksnctf7();
let src = std::fs::read_to_string("program.cpp").unwrap();
let tokens = filter_source(&src);
let instrs = parse_ws(&tokens);

println!("=================");

// ✅ 正しいPIN
let ok = run_vm(&instrs, "33355524");
let ok = ok.trim().to_string();

// FLAG部分をマスクして表示
// 出力例: "PIN: OK\nFLAG_xxxx" → FLAG_以降をマスク
let masked = ok.lines().map(|line| {
    if line.starts_with("FLAG_") {
        let visible = &line[..6]; // "FLAG_X" の先頭6文字だけ表示
        format!("{}**************", visible)
    } else {
        line.to_string()
    }
}).collect::<Vec<_>>().join("\n");

println!("[PIN = 33355524] →\n{}", masked);

// FLAG を flag.txt に出力
let flag_line = ok.lines()
    .find(|l| l.starts_with("FLAG_"))
    .unwrap_or("")
    .to_string(); 
std::fs::write("flag.txt", flag_line).unwrap();
println!("FLAG を flag.txt に保存しました");

// ❌ 誤ったPIN
let ng = run_vm(&instrs, "12345678");
println!("[PIN = 12345678] → {}", ng.trim());

=== label_map ===
  key="\t\n"  idx=22
=== instrs ===
  [00] Push(80)
  [01] Putc
  [02] Push(73)
  [03] Putc
  [04] Push(78)
  [05] Putc
  [06] Push(58)
  [07] Putc
  [08] Push(32)
  [09] Putc
  [10] Push(0)
  [11] Readi
  [12] Push(0)
  [13] Load
  [14] Push(33355524)
  [15] Sub
  [16] Jz("\t\n")
  [17] Push(78)
  [18] Putc
  [19] Push(79)
  [20] Putc
  [21] End
  [22] Label("\t\n")
  [23] Push(79)
  [24] Putc
  [25] Push(75)
  [26] Putc
  [27] Push(10)
  [28] Putc
  [29] Push(0)
  [30] Load
  [31] Push(33355454)
  [32] Sub
  [33] Dup
  [34] Putc
  [35] Push(6)
  [36] Add
  [37] Dup
  [38] Putc
  [39] Push(11)
  [40] Sub
  [41] Dup
  [42] Putc
  [43] Push(6)
  [44] Add
  [45] Dup
  [46] Putc
  [47] Push(24)
  [48] Add
  [49] Dup
  [50] Putc
  [51] Push(26)
  [52] Sub
  [53] Dup
  [54] Putc
  [55] Push(40)
  [56] Add
  [57] Dup
  [58] Putc
  [59] Push(25)
  [60] Sub
  [61] Dup
  [62] Putc
  [63] Push(36)
  [64] Add
  [65] Dup
  [66] Putc
  [67] Push(66)
  [68] Sub
  [69] Dup
  [70] Pu